# 1. Environment and Configuration

## 1.1 Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 1.2 Clone Repository and Install Dependencies

In [ ]:
import subprocess
import sys
from pathlib import Path

from google.colab import userdata

REPOSITORY_URL = (
    "https://github.com/FillipusAditya/"
    "mask-guided-lung-nodule-xai.git"
)
REPOSITORY_BRANCH = "main"
PROJECT_ROOT = Path("/content/mask-guided-lung-nodule-xai")

if not PROJECT_ROOT.is_dir():
    try:
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception as error:
        raise RuntimeError(
            "Add a GITHUB_TOKEN secret in Google Colab before cloning."
        ) from error
    clone_url = REPOSITORY_URL.replace(
        "https://",
        f"https://x-access-token:{github_token}@",
    )
    result = subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            REPOSITORY_BRANCH,
            "--single-branch",
            clone_url,
            str(PROJECT_ROOT),
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        safe_error = result.stderr.replace(github_token, "***")
        raise RuntimeError(safe_error)
    subprocess.run(
        [
            "git",
            "-C",
            str(PROJECT_ROOT),
            "remote",
            "set-url",
            "origin",
            REPOSITORY_URL,
        ],
        check=True,
    )

packages = [
    "albumentations>=2.0,<3.0",
    "opencv-python-headless",
    "pandas",
    "matplotlib",
    "scikit-learn",
    "tqdm",
    "zennit==0.5.1",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True,
)

print(f"Repository: {PROJECT_ROOT}")

## 1.3 Analysis Configuration

In [ ]:
EXPERIMENT_ID = "fa028196-fd4c-441f-a5d3-3efb9707e7f9"
DRIVE_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mask-guided-lung-nodule-xai"
)
DRIVE_MODEL_DIR = (
    DRIVE_PROJECT_ROOT
    / "experiment_results"
    / EXPERIMENT_ID
    / "classification/multistage_direct_guided_cv_resnet50"
)
DRIVE_OUTPUT_DIR = (
    DRIVE_PROJECT_ROOT
    / "experiment_results"
    / EXPERIMENT_ID
    / "classification/1st_review_responses_multistage_direct_guided_cv_resnet50"
)

ARCHIVE_NAME = f"{EXPERIMENT_ID}_guided_resnet50_parenchyma.tar.gz"
DRIVE_ARCHIVE_PATH = DRIVE_PROJECT_ROOT / ARCHIVE_NAME
LOCAL_ARCHIVE_PATH = Path("/content") / ARCHIVE_NAME
EXTRACTION_ROOT = Path("/content/aaiml_1st_response_data")
DATASET_ROOT = EXTRACTION_ROOT / "000_dataset_v2/_segmentation_dataset"
METADATA_PATH = DATASET_ROOT / "004_classification_cv_5fold_seed42.csv"
PROBABILITY_ROOT = (
    EXTRACTION_ROOT
    / "experiment_results"
    / EXPERIMENT_ID
    / "segmentation/unet/inference/probability_npy"
)
LOCAL_MODEL_DIR = Path("/content/aaiml_1st_response_multistage_direct_guided_cv_resnet50")

COPY_ARCHIVE_TO_LOCAL = True
COPY_MODEL_INPUTS_TO_LOCAL = True
FORCE_EXTRACT = False
FORCE_MODEL_COPY = False
RUN_INFERENCE = True
RUN_XAI_EVALUATION = True
DEVICE = "auto"
INFERENCE_BATCH_SIZE = 64
INFERENCE_NUM_WORKERS = 2
CLASSIFICATION_THRESHOLD = 0.5
BOOTSTRAP_ITERATIONS = 10_000
BOOTSTRAP_SEED = 42
CONFIDENCE_LEVEL = 0.95
FIGURE_DPI = 300
TOP_K_PERCENTAGES = (1.0, 5.0, 10.0, 20.0)
DELETION_PERCENTAGES = (5.0, 10.0, 20.0, 30.0, 40.0, 50.0)
METHODS = ("gradcam", "lrp")
LAYERS = ("layer1", "layer2", "layer3", "layer4")
SOURCE_SCOPES = {
    "combined": None,
    "lidc_idri": "LIDC-IDRI",
    "lndb": "LNDb",
}

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Model input: {DRIVE_MODEL_DIR}")
print(f"Output: {DRIVE_OUTPUT_DIR}")

## 1.4 Figure Style

In [ ]:
import json
import math
import shutil
from importlib import import_module

from IPython.display import Image, display
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
import torch
from tqdm.auto import tqdm

tinos_path = Path("/usr/share/fonts/truetype/croscore/Tinos-Regular.ttf")
if tinos_path.is_file():
    font_manager.fontManager.addfont(tinos_path)
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
font_candidates = ("Times New Roman", "Tinos", "Liberation Serif")
FONT_FAMILY = next(
    (name for name in font_candidates if name in available_fonts),
    "serif",
)
plt.rcParams.update(
    {
        "font.family": FONT_FAMILY,
        "font.size": 8,
        "axes.labelsize": 8,
        "axes.titlesize": 8,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "axes.facecolor": "white",
        "figure.facecolor": "white",
        "savefig.facecolor": "white",
    }
)
MODEL_COLOR = "#D55E00"
GRID_COLOR = "#D0D0D0"


def style_axis(axis, grid_axis="both"):
    axis.grid(True, axis=grid_axis, color=GRID_COLOR, linewidth=0.35)
    axis.set_axisbelow(True)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.spines["left"].set_linewidth(0.6)
    axis.spines["bottom"].set_linewidth(0.6)
    axis.tick_params(width=0.6, length=2.5)


def save_figure(figure, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(
        output_path,
        dpi=FIGURE_DPI,
        bbox_inches="tight",
        pad_inches=0.03,
        facecolor="white",
    )
    plt.close(figure)
    return output_path


print(f"Font: {FONT_FAMILY}")

# 2. Input Preparation

## 2.1 Copy Original Model Artifacts

In [ ]:
def copy_file(source, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if FORCE_MODEL_COPY or not destination.is_file():
        shutil.copy2(source, destination)


if not DRIVE_MODEL_DIR.is_dir():
    raise FileNotFoundError(f"Model directory not found: {DRIVE_MODEL_DIR}")

if COPY_MODEL_INPUTS_TO_LOCAL:
    LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    for fold in range(5):
        copy_file(
            DRIVE_MODEL_DIR / f"fold_{fold}/best_model.pt",
            LOCAL_MODEL_DIR / f"fold_{fold}/best_model.pt",
        )
    copy_file(
        DRIVE_MODEL_DIR / "run_config.json",
        LOCAL_MODEL_DIR / "run_config.json",
    )
    copy_file(
        DRIVE_MODEL_DIR / "multistage_direct_guided_cv_resnet50.json",
        LOCAL_MODEL_DIR / "multistage_direct_guided_cv_resnet50.json",
    )
    copy_file(
        DRIVE_MODEL_DIR / "test/test_predictions.csv",
        LOCAL_MODEL_DIR / "test/test_predictions.csv",
    )
    for directory_name in ("gradcam_npy", "lrp_npy"):
        source = DRIVE_MODEL_DIR / "test" / directory_name
        destination = LOCAL_MODEL_DIR / "test" / directory_name
        if FORCE_MODEL_COPY and destination.exists():
            shutil.rmtree(destination)
        if not destination.is_dir():
            shutil.copytree(source, destination)
else:
    LOCAL_MODEL_DIR = DRIVE_MODEL_DIR

print(f"Local model directory: {LOCAL_MODEL_DIR}")

## 2.2 Extract Original Dataset

In [ ]:
if not DRIVE_ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f"Archive not found: {DRIVE_ARCHIVE_PATH}")

archive_path = DRIVE_ARCHIVE_PATH
if COPY_ARCHIVE_TO_LOCAL:
    if not LOCAL_ARCHIVE_PATH.is_file():
        shutil.copy2(DRIVE_ARCHIVE_PATH, LOCAL_ARCHIVE_PATH)
    archive_path = LOCAL_ARCHIVE_PATH

if FORCE_EXTRACT and EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)
if not METADATA_PATH.is_file():
    EXTRACTION_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["tar", "-xzf", str(archive_path), "-C", str(EXTRACTION_ROOT)],
        check=True,
    )

if not METADATA_PATH.is_file():
    raise FileNotFoundError(f"Metadata not found after extraction: {METADATA_PATH}")
if not PROBABILITY_ROOT.is_dir():
    raise FileNotFoundError(
        f"Probability-map directory not found: {PROBABILITY_ROOT}"
    )

print(f"Dataset root: {DATASET_ROOT}")
print(f"Metadata: {METADATA_PATH}")
print(f"Probability maps: {PROBABILITY_ROOT}")

## 2.3 Validate Inputs

In [ ]:
required_paths = [
    LOCAL_MODEL_DIR / "run_config.json",
    LOCAL_MODEL_DIR / "multistage_direct_guided_cv_resnet50.json",
    LOCAL_MODEL_DIR / "test/test_predictions.csv",
    METADATA_PATH,
]
required_paths.extend(
    LOCAL_MODEL_DIR / f"fold_{fold}/best_model.pt"
    for fold in range(5)
)
missing_paths = [path for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(
        "Missing required files:\n"
        + "\n".join(str(path) for path in missing_paths)
    )

cached_predictions = pd.read_csv(
    LOCAL_MODEL_DIR / "test/test_predictions.csv"
)
expected_maps = len(cached_predictions) * len(METHODS) * len(LAYERS)
available_maps = sum(
    len(list((LOCAL_MODEL_DIR / "test" / f"{method}_npy" / layer).glob("*.npy")))
    for method in METHODS
    for layer in LAYERS
)
if available_maps != expected_maps:
    raise ValueError(
        f"Expected {expected_maps:,} XAI maps, found {available_maps:,}."
    )

test_probability_maps = sum(
    (PROBABILITY_ROOT / Path(str(filename)).name).is_file()
    for filename in cached_predictions["filename"]
)
if test_probability_maps != len(cached_predictions):
    raise ValueError(
        f"Expected {len(cached_predictions):,} test probability maps, "
        f"found {test_probability_maps:,}."
    )

print(f"Cached test slices: {len(cached_predictions):,}")
print(f"XAI maps: {available_maps:,}")
print(f"Test probability maps: {test_probability_maps:,}")

# 3. Held-Out Test Inference

## 3.1 Load Five-Fold Ensemble

In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
test_module = import_module(
    "003_classification.multistage_direct_guided_cv_resnet50.test"
)

device_name = DEVICE
if device_name == "auto":
    device_name = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(device_name)

fold_config, model_options, config_sources = test_module.load_run_config(
    LOCAL_MODEL_DIR
)
fold_config = test_module.apply_data_overrides(
    config=fold_config,
    dataset_root=DATASET_ROOT,
    metadata_path=METADATA_PATH,
    probability_root=PROBABILITY_ROOT,
)
test_loader = test_module.build_test_loader(
    config=fold_config,
    batch_size=INFERENCE_BATCH_SIZE,
    num_workers=INFERENCE_NUM_WORKERS,
    max_samples=None,
)
models = test_module.load_models(
    LOCAL_MODEL_DIR, fold_config, model_options, device
)

print(f"Device: {device}")
print(f"Models: {len(models)}")
print(f"Held-out slices: {len(test_loader.dataset):,}")

## 3.2 Run One Held-Out Test Inference

In [ ]:
def run_ensemble_inference(model_list, loader, inference_device):
    records = []
    sample_index = 0
    with torch.inference_mode():
        for inputs, probability_maps, labels in tqdm(
            loader, desc="Held-out inference"
        ):
            inputs = inputs.to(inference_device, non_blocking=True)
            probability_maps = probability_maps.to(
                inference_device, non_blocking=True
            )
            fold_probabilities = [
                torch.softmax(model(inputs, probability_maps), dim=1)
                for model in model_list
            ]
            probabilities = torch.stack(fold_probabilities).mean(dim=0)
            predictions = (
                probabilities[:, 1] >= CLASSIFICATION_THRESHOLD
            ).long()

            for batch_index in range(len(labels)):
                row = dict(loader.dataset.metadata.iloc[sample_index])
                prediction = int(predictions[batch_index])
                row.update(
                    {
                        "true_index": int(labels[batch_index]),
                        "predicted_index": prediction,
                        "predicted_class": (
                            "malignant" if prediction == 1 else "benign"
                        ),
                        "probability_benign": float(
                            probabilities[batch_index, 0]
                        ),
                        "probability_malignant": float(
                            probabilities[batch_index, 1]
                        ),
                    }
                )
                records.append(row)
                sample_index += 1
    return pd.DataFrame(records)


prediction_directory = DRIVE_OUTPUT_DIR / "predictions"
prediction_directory.mkdir(parents=True, exist_ok=True)
slice_prediction_path = prediction_directory / "slice_predictions.csv"
if RUN_INFERENCE or not slice_prediction_path.is_file():
    slice_predictions = run_ensemble_inference(models, test_loader, device)
    slice_predictions.to_csv(slice_prediction_path, index=False)
else:
    slice_predictions = pd.read_csv(slice_prediction_path)

cached_comparison = cached_predictions[
    ["filename", "predicted_index", "probability_malignant"]
].rename(
    columns={
        "predicted_index": "cached_predicted_index",
        "probability_malignant": "cached_probability_malignant",
    }
)
inference_consistency = slice_predictions.merge(
    cached_comparison, on="filename", validate="one_to_one"
)
inference_consistency["prediction_matches"] = (
    inference_consistency["predicted_index"]
    == inference_consistency["cached_predicted_index"]
)
inference_consistency["absolute_probability_error"] = (
    inference_consistency["probability_malignant"]
    - inference_consistency["cached_probability_malignant"]
).abs()
inference_consistency.to_csv(
    prediction_directory / "inference_consistency.csv", index=False
)

print(f"Saved: {slice_prediction_path}")
print(
    "Changed predicted classes: "
    f"{int((~inference_consistency['prediction_matches']).sum())}"
)

## 3.3 Build Nodule-Level Predictions

In [ ]:
def aggregate_nodule_predictions(frame):
    records = []
    for nodule_id, group in frame.groupby("cv_nodule_id", sort=True):
        true_values = group["true_index"].astype(int).unique()
        if len(true_values) != 1:
            raise ValueError(f"Inconsistent label for {nodule_id}")
        benign_votes = int((group["predicted_index"] == 0).sum())
        malignant_votes = int((group["predicted_index"] == 1).sum())
        mean_malignant = float(group["probability_malignant"].mean())
        tied = benign_votes == malignant_votes
        if tied:
            prediction = int(mean_malignant >= CLASSIFICATION_THRESHOLD)
        else:
            prediction = int(malignant_votes > benign_votes)
        records.append(
            {
                "dataset": group["dataset"].iloc[0],
                "patient_id": group["patient_id"].iloc[0],
                "cv_group_id": group["cv_group_id"].iloc[0],
                "nodule_id": group["nodule_id"].iloc[0],
                "cv_nodule_id": nodule_id,
                "label": group["label"].iloc[0],
                "target": int(true_values[0]),
                "num_slices": len(group),
                "benign_votes": benign_votes,
                "malignant_votes": malignant_votes,
                "malignant_vote_fraction": malignant_votes / len(group),
                "mean_probability_benign": float(
                    group["probability_benign"].mean()
                ),
                "mean_probability_malignant": mean_malignant,
                "tie_break_used": tied,
                "prediction": prediction,
                "predicted_class": (
                    "malignant" if prediction == 1 else "benign"
                ),
                "correct": prediction == int(true_values[0]),
            }
        )
    return pd.DataFrame(records)


nodule_predictions = aggregate_nodule_predictions(slice_predictions)
nodule_prediction_path = prediction_directory / "nodule_predictions.csv"
nodule_predictions.to_csv(nodule_prediction_path, index=False)
display(nodule_predictions.head())
print(f"Nodules: {len(nodule_predictions):,}")

for model in models:
    model.cpu()
models.clear()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 4. Source-Specific Classification Evaluation

## 4.1 Classification Evaluation Functions

In [ ]:
METRIC_NAMES = (
    "accuracy",
    "sensitivity",
    "specificity",
    "precision",
    "f1_score",
)


def safe_ratio(numerator, denominator):
    return float(numerator / denominator) if denominator else np.nan


def binary_metrics(targets, predictions):
    targets = np.asarray(targets, dtype=int)
    predictions = np.asarray(predictions, dtype=int)
    tn = int(np.sum((targets == 0) & (predictions == 0)))
    fp = int(np.sum((targets == 0) & (predictions == 1)))
    fn = int(np.sum((targets == 1) & (predictions == 0)))
    tp = int(np.sum((targets == 1) & (predictions == 1)))
    return {
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "accuracy": safe_ratio(tp + tn, len(targets)),
        "sensitivity": safe_ratio(tp, tp + fn),
        "specificity": safe_ratio(tn, tn + fp),
        "precision": safe_ratio(tp, tp + fp),
        "f1_score": safe_ratio(2 * tp, 2 * tp + fp + fn),
    }


def standardize_predictions(frame, level):
    result = frame.copy()
    if level == "slice":
        result["target"] = result["true_index"].astype(int)
        result["prediction"] = result["predicted_index"].astype(int)
        result["probability"] = result["probability_malignant"].astype(float)
    else:
        result["target"] = result["target"].astype(int)
        result["prediction"] = result["prediction"].astype(int)
        result["probability"] = result["mean_probability_malignant"].astype(float)
    result["unique_patient_id"] = (
        result["dataset"].astype(str)
        + "::"
        + result["patient_id"].astype(str)
    )
    return result


def plot_confusion_matrix_file(matrix, output_path, title):
    figure, axis = plt.subplots(figsize=(3.15, 2.75))
    image = axis.imshow(matrix, cmap="Blues", vmin=0)
    threshold = matrix.max() / 2
    for row in range(2):
        for column in range(2):
            axis.text(
                column,
                row,
                f"{matrix[row, column]:,}",
                ha="center",
                va="center",
                color="white" if matrix[row, column] > threshold else "black",
            )
    axis.set_xticks([0, 1], ["Benign", "Malignant"])
    axis.set_yticks([0, 1], ["Benign", "Malignant"])
    axis.set_xlabel("Predicted label")
    axis.set_ylabel("True label")
    axis.set_title(title)
    axis.grid(False)
    figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    figure.tight_layout()
    return save_figure(figure, output_path)


def plot_roc_file(false_positive_rate, true_positive_rate, auc_value, output_path):
    figure, axis = plt.subplots(figsize=(3.35, 3.10))
    axis.plot(
        false_positive_rate,
        true_positive_rate,
        color=MODEL_COLOR,
        linewidth=1.25,
        label=f"Direct-Guided ResNet-50 (AUC = {auc_value:.3f})",
    )
    axis.plot([0, 1], [0, 1], color="#606060", linestyle="--", linewidth=0.8)
    axis.set(xlim=(0, 1), ylim=(0, 1), xlabel="False Positive Rate", ylabel="True Positive Rate")
    style_axis(axis)
    axis.legend(loc="lower right", frameon=False)
    figure.tight_layout()
    return save_figure(figure, output_path)


def plot_pr_file(recall, precision, ap_value, prevalence, output_path):
    figure, axis = plt.subplots(figsize=(3.35, 3.10))
    axis.plot(
        recall,
        precision,
        color=MODEL_COLOR,
        linewidth=1.25,
        label=f"Direct-Guided ResNet-50 (AP = {ap_value:.3f})",
    )
    axis.axhline(prevalence, color="#606060", linestyle="--", linewidth=0.8, label=f"Prevalence = {prevalence:.3f}")
    axis.set(xlim=(0, 1), ylim=(0, 1), xlabel="Recall", ylabel="Precision")
    style_axis(axis)
    axis.legend(loc="lower left", frameon=False)
    figure.tight_layout()
    return save_figure(figure, output_path)

## 4.2 Export Classification Results

In [ ]:
classification_results = {}
classification_root = DRIVE_OUTPUT_DIR / "classification"
level_frames = {
    "slice": standardize_predictions(slice_predictions, "slice"),
    "nodule": standardize_predictions(nodule_predictions, "nodule"),
}

for scope_name, dataset_name in SOURCE_SCOPES.items():
    for level, all_predictions in level_frames.items():
        selected = all_predictions.copy()
        if dataset_name is not None:
            selected = selected[selected["dataset"].eq(dataset_name)].copy()
        output_directory = classification_root / scope_name / level
        output_directory.mkdir(parents=True, exist_ok=True)
        selected.to_csv(output_directory / "predictions.csv", index=False)

        metrics = binary_metrics(selected["target"], selected["prediction"])
        metric_table = pd.DataFrame(
            [
                {
                    "dataset_scope": scope_name,
                    "evaluation_level": level,
                    "samples": len(selected),
                    "patients": selected["unique_patient_id"].nunique(),
                    **{name: round(metrics[name] * 100, 2) for name in METRIC_NAMES},
                }
            ]
        )
        metric_table.to_csv(output_directory / "metrics.csv", index=False)

        matrix = np.array(
            [[metrics["tn"], metrics["fp"]], [metrics["fn"], metrics["tp"]]]
        )
        matrix_table = pd.DataFrame(
            matrix,
            index=["benign", "malignant"],
            columns=["benign", "malignant"],
        )
        matrix_table.to_csv(
            output_directory / "confusion_matrix.csv",
            index_label="true_label",
        )
        confusion_path = plot_confusion_matrix_file(
            matrix,
            output_directory / "confusion_matrix.png",
            f"{scope_name.replace('_', ' ').title()} - {level.title()}",
        )

        if selected["target"].nunique() != 2:
            raise ValueError(f"{scope_name}/{level} does not contain both classes.")
        fpr, tpr, roc_thresholds = roc_curve(
            selected["target"], selected["probability"]
        )
        precision, recall, pr_thresholds = precision_recall_curve(
            selected["target"], selected["probability"]
        )
        auc_value = roc_auc_score(selected["target"], selected["probability"])
        ap_value = average_precision_score(selected["target"], selected["probability"])
        pd.DataFrame(
            {"threshold": roc_thresholds, "false_positive_rate": fpr, "true_positive_rate": tpr}
        ).to_csv(output_directory / "roc_curve.csv", index=False)
        pd.DataFrame(
            {
                "threshold": np.append(pr_thresholds, np.nan),
                "recall": recall,
                "precision": precision,
            }
        ).to_csv(output_directory / "pr_curve.csv", index=False)
        pd.DataFrame(
            [
                {
                    "dataset_scope": scope_name,
                    "evaluation_level": level,
                    "samples": len(selected),
                    "positive_samples": int(selected["target"].sum()),
                    "negative_samples": int((selected["target"] == 0).sum()),
                    "roc_auc": auc_value,
                    "average_precision": ap_value,
                }
            ]
        ).to_csv(output_directory / "roc_pr_summary.csv", index=False)
        roc_path = plot_roc_file(
            fpr, tpr, auc_value, output_directory / "roc_curve.png"
        )
        pr_path = plot_pr_file(
            recall,
            precision,
            ap_value,
            float(selected["target"].mean()),
            output_directory / "pr_curve.png",
        )
        classification_results[(scope_name, level)] = {
            "metrics": metric_table,
            "confusion": confusion_path,
            "roc": roc_path,
            "pr": pr_path,
        }

print(f"Classification outputs: {classification_root}")

## 4.3 Classification Tables and Figures

In [ ]:
for scope_name in SOURCE_SCOPES:
    for level in ("slice", "nodule"):
        result = classification_results[(scope_name, level)]
        print(f"{scope_name} - {level}")
        display(result["metrics"])
        display(Image(filename=str(result["confusion"])))
        display(Image(filename=str(result["roc"])))
        display(Image(filename=str(result["pr"])))

# 5. Patient-Clustered Confidence Intervals

## 5.1 Patient-Clustered Bootstrap Functions

In [ ]:
def patient_clustered_bootstrap(frame):
    patient_rows = []
    for patient_id, group in frame.groupby("unique_patient_id", sort=True):
        metrics = binary_metrics(group["target"], group["prediction"])
        patient_rows.append(
            [metrics["tn"], metrics["fp"], metrics["fn"], metrics["tp"]]
        )
    patient_counts = np.asarray(patient_rows, dtype=int)
    number_of_patients = len(patient_counts)
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    sampled_indices = rng.integers(
        0,
        number_of_patients,
        size=(BOOTSTRAP_ITERATIONS, number_of_patients),
    )
    counts = patient_counts[sampled_indices].sum(axis=1)
    tn, fp, fn, tp = counts.T.astype(float)

    def divide(numerator, denominator):
        return np.divide(
            numerator,
            denominator,
            out=np.full_like(numerator, np.nan, dtype=float),
            where=denominator != 0,
        )

    distributions = {
        "accuracy": divide(tp + tn, tp + tn + fp + fn),
        "sensitivity": divide(tp, tp + fn),
        "specificity": divide(tn, tn + fp),
        "precision": divide(tp, tp + fp),
        "f1_score": divide(2 * tp, 2 * tp + fp + fn),
    }
    bootstrap_frame = pd.DataFrame(distributions)
    bootstrap_frame.insert(0, "iteration", np.arange(1, BOOTSTRAP_ITERATIONS + 1))

    point_metrics = binary_metrics(frame["target"], frame["prediction"])
    alpha = 1.0 - CONFIDENCE_LEVEL
    records = []
    for metric_name in METRIC_NAMES:
        values = bootstrap_frame[metric_name].dropna().to_numpy()
        lower, upper = np.quantile(values, [alpha / 2, 1 - alpha / 2])
        records.append(
            {
                "metric": metric_name,
                "estimate": point_metrics[metric_name] * 100,
                "ci_lower": lower * 100,
                "ci_upper": upper * 100,
                "valid_iterations": len(values),
                "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
                "confidence_level": CONFIDENCE_LEVEL,
                "patients": number_of_patients,
            }
        )
    return bootstrap_frame, pd.DataFrame(records)


def plot_confidence_intervals(summary, output_path):
    positions = np.arange(len(summary))
    estimates = summary["estimate"].to_numpy()
    lower = estimates - summary["ci_lower"].to_numpy()
    upper = summary["ci_upper"].to_numpy() - estimates
    figure, axis = plt.subplots(figsize=(4.20, 2.75))
    axis.errorbar(
        estimates,
        positions,
        xerr=np.vstack([lower, upper]),
        fmt="o",
        color=MODEL_COLOR,
        ecolor="#404040",
        elinewidth=1.0,
        capsize=3,
        markersize=4,
    )
    axis.set_yticks(positions, [name.replace('_', ' ').title() for name in summary["metric"]])
    axis.set_xlabel("Performance (%)")
    axis.invert_yaxis()
    style_axis(axis, grid_axis="x")
    figure.tight_layout()
    return save_figure(figure, output_path)

## 5.2 Export Confidence Intervals

In [ ]:
confidence_interval_results = {}
for scope_name, dataset_name in SOURCE_SCOPES.items():
    for level, all_predictions in level_frames.items():
        selected = all_predictions.copy()
        if dataset_name is not None:
            selected = selected[selected["dataset"].eq(dataset_name)].copy()
        output_directory = classification_root / scope_name / level
        bootstrap_frame, summary = patient_clustered_bootstrap(selected)
        bootstrap_frame.to_csv(
            output_directory / "patient_clustered_bootstrap.csv", index=False
        )
        summary = summary.round(
            {"estimate": 2, "ci_lower": 2, "ci_upper": 2}
        )
        summary.to_csv(
            output_directory / "patient_clustered_ci_summary.csv", index=False
        )
        figure_path = plot_confidence_intervals(
            summary, output_directory / "patient_clustered_ci.png"
        )
        confidence_interval_results[(scope_name, level)] = (summary, figure_path)

print("Confidence intervals saved.")

## 5.3 Confidence Interval Tables and Figures

In [ ]:
for scope_name in SOURCE_SCOPES:
    for level in ("slice", "nodule"):
        summary, figure_path = confidence_interval_results[(scope_name, level)]
        print(f"{scope_name} - {level}")
        display(summary)
        display(Image(filename=str(figure_path)))

# 6. XAI Plausibility and Nodule Coverage

## 6.1 Run Combined Plausibility Evaluation

In [ ]:
raw_xai_root = DRIVE_OUTPUT_DIR / "raw_xai"
raw_plausibility_dir = raw_xai_root / "plausibility"
raw_plausibility_path = raw_plausibility_dir / "plausibility_slice.csv"
if RUN_XAI_EVALUATION or not raw_plausibility_path.is_file():
    subprocess.run(
        [
            sys.executable,
            str(PROJECT_ROOT / "004_evaluation/evaluate_xai_plausibility.py"),
            "--experiment-dir",
            str(LOCAL_MODEL_DIR.parent),
            "--dataset-root",
            str(DATASET_ROOT),
            "--probability-root",
            str(PROBABILITY_ROOT),
            "--output-dir",
            str(raw_plausibility_dir),
            "--model-dir",
            str(LOCAL_MODEL_DIR),
            "--top-k",
            *[str(value) for value in TOP_K_PERCENTAGES],
            "--primary-top-k",
            "10",
        ],
        check=True,
    )

plausibility_slice = pd.read_csv(raw_plausibility_path)
plausibility_slice["nodule_coverage"] = (
    plausibility_slice["intersection_pixels"]
    / plausibility_slice["ground_truth_pixels"]
)
print(f"Plausibility rows: {len(plausibility_slice):,}")

## 6.2 Export Source-Specific Plausibility Results

In [ ]:
def plot_plausibility_summary(summary, output_path):
    labels = [
        f"{method.upper()} L{layer[-1]}"
        for method in METHODS
        for layer in LAYERS
    ]
    colors = ["#000000", "#0067C5", "#E66100", "#7B2CBF"]
    figure, axes = plt.subplots(1, 2, figsize=(7.16, 2.85))
    for top_k, marker in zip(TOP_K_PERCENTAGES, ("o", "s", "^", "D")):
        subset = summary[summary["top_k_percent"].eq(top_k)].copy()
        subset["order"] = pd.Categorical(
            subset["xai_method"] + "_" + subset["layer"],
            categories=[f"{method}_{layer}" for method in METHODS for layer in LAYERS],
            ordered=True,
        )
        subset = subset.sort_values("order")
        axes[0].plot(
            range(len(labels)), subset["mean_iou_across_nodules"],
            marker=marker, linewidth=1.1, markersize=3, label=f"Top {top_k:g}%",
        )
        axes[1].plot(
            range(len(labels)), subset["mean_coverage_across_nodules"] * 100,
            marker=marker, linewidth=1.1, markersize=3, label=f"Top {top_k:g}%",
        )
    axes[0].set_ylabel("IoU")
    axes[1].set_ylabel("Nodule Coverage (%)")
    for axis in axes:
        axis.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
        style_axis(axis)
    axes[1].legend(loc="best", frameon=False, ncol=2)
    figure.subplots_adjust(left=0.08, right=0.995, bottom=0.28, top=0.98, wspace=0.25)
    return save_figure(figure, output_path)


plausibility_results = {}
plausibility_root = DRIVE_OUTPUT_DIR / "xai_plausibility"
for scope_name, dataset_name in SOURCE_SCOPES.items():
    selected = plausibility_slice.copy()
    if dataset_name is not None:
        selected = selected[selected["dataset"].eq(dataset_name)].copy()
    output_directory = plausibility_root / scope_name
    output_directory.mkdir(parents=True, exist_ok=True)
    selected.to_csv(output_directory / "plausibility_slice.csv", index=False)

    nodule = (
        selected.groupby(
            ["model", "xai_method", "layer", "top_k_percent", "dataset", "patient_id", "nodule_id"],
            as_index=False,
        )
        .agg(
            number_of_slices=("filename", "nunique"),
            mean_slice_iou=("iou", "mean"),
            mean_slice_coverage=("nodule_coverage", "mean"),
        )
    )
    nodule.to_csv(output_directory / "plausibility_nodule.csv", index=False)
    summary = (
        nodule.groupby(["model", "xai_method", "layer", "top_k_percent"], as_index=False)
        .agg(
            number_of_nodules=("nodule_id", "count"),
            number_of_slices=("number_of_slices", "sum"),
            mean_iou_across_nodules=("mean_slice_iou", "mean"),
            std_iou_across_nodules=("mean_slice_iou", "std"),
            mean_coverage_across_nodules=("mean_slice_coverage", "mean"),
            std_coverage_across_nodules=("mean_slice_coverage", "std"),
        )
    )
    summary.to_csv(output_directory / "plausibility_summary.csv", index=False)
    figure_path = plot_plausibility_summary(
        summary, output_directory / "plausibility_and_nodule_coverage.png"
    )
    plausibility_results[scope_name] = (summary, figure_path)

print(f"Plausibility outputs: {plausibility_root}")

## 6.3 Plausibility Tables and Figures

In [ ]:
for scope_name, (summary, figure_path) in plausibility_results.items():
    print(scope_name)
    display(summary.round(4))
    display(Image(filename=str(figure_path)))

# 7. XAI Fidelity

## 7.1 Run Combined Fidelity Evaluation

In [ ]:
raw_fidelity_root = raw_xai_root / "fidelity"
raw_fidelity_model_dir = raw_fidelity_root / LOCAL_MODEL_DIR.name
raw_fidelity_curve_path = raw_fidelity_model_dir / "fidelity_curve.csv"
if RUN_XAI_EVALUATION or not raw_fidelity_curve_path.is_file():
    subprocess.run(
        [
            sys.executable,
            str(PROJECT_ROOT / "004_evaluation/evaluate_xai_fidelity.py"),
            "--experiment-dir",
            str(LOCAL_MODEL_DIR.parent),
            "--model-dir",
            str(LOCAL_MODEL_DIR),
            "--dataset-root",
            str(DATASET_ROOT),
            "--output-dir",
            str(raw_fidelity_root),
            "--methods",
            *METHODS,
            "--layers",
            *LAYERS,
            "--blur-config",
            "weak",
            "medium",
            "strong",
            "--deletion-percentages",
            *[str(value) for value in DELETION_PERCENTAGES],
            "--inference-batch-size",
            str(INFERENCE_BATCH_SIZE),
            "--device",
            DEVICE,
        ],
        check=True,
    )

fidelity_curve = pd.read_csv(raw_fidelity_curve_path)
fidelity_nodule = pd.read_csv(raw_fidelity_model_dir / "fidelity_nodule.csv")
print(f"Fidelity curve rows: {len(fidelity_curve):,}")

## 7.2 Export Source-Specific Fidelity Results

In [ ]:
BLUR_STYLES = {
    "weak": {"color": "#0072B2", "linestyle": "-", "marker": "o"},
    "medium": {"color": "#D55E00", "linestyle": "--", "marker": "s"},
    "strong": {"color": "#009E73", "linestyle": "-.", "marker": "^"},
}


def plot_fidelity_curves(curve_points, method, output_path):
    figure, axes = plt.subplots(1, 4, figsize=(7.16, 2.25), sharex=True, sharey=True)
    for axis, layer in zip(axes, LAYERS):
        layer_frame = curve_points[
            curve_points["xai_method"].eq(method)
            & curve_points["layer"].eq(layer)
        ]
        for blur_name, style in BLUR_STYLES.items():
            values = layer_frame[layer_frame["blur_name"].eq(blur_name)].sort_values("deletion_percent")
            axis.plot(
                values["deletion_percent"],
                values["mean_target_probability"],
                label=blur_name.title(),
                color=style["color"],
                linestyle=style["linestyle"],
                marker=style["marker"],
                linewidth=1.15,
                markersize=3,
                markerfacecolor="white",
            )
        axis.set_title(f"L{layer[-1]}")
        axis.set_xticks([0, 10, 20, 30, 40, 50])
        axis.set_xlim(-1, 51)
        axis.set_ylim(0, 1)
        style_axis(axis)
    handles, labels = axes[0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.02))
    figure.supxlabel("Deleted Attribution (%)", y=0.03)
    figure.supylabel("Target-Class Probability", x=0.01)
    figure.subplots_adjust(left=0.08, right=0.995, bottom=0.22, top=0.80, wspace=0.15)
    return save_figure(figure, output_path)


fidelity_results = {}
fidelity_root = DRIVE_OUTPUT_DIR / "xai_fidelity"
for scope_name, dataset_name in SOURCE_SCOPES.items():
    selected_curve = fidelity_curve.copy()
    selected_nodule = fidelity_nodule.copy()
    if dataset_name is not None:
        selected_curve = selected_curve[selected_curve["dataset"].eq(dataset_name)].copy()
        selected_nodule = selected_nodule[selected_nodule["dataset"].eq(dataset_name)].copy()
    output_directory = fidelity_root / scope_name
    output_directory.mkdir(parents=True, exist_ok=True)

    nodule_curves = (
        selected_curve.groupby(
            ["xai_method", "layer", "blur_name", "blur_sigma", "blur_kernel_size", "nodule_id", "deletion_percent"],
            as_index=False,
        )
        .agg(target_probability=("perturbed_probability", "mean"))
    )
    curve_points = (
        nodule_curves.groupby(
            ["xai_method", "layer", "blur_name", "blur_sigma", "blur_kernel_size", "deletion_percent"],
            as_index=False,
        )
        .agg(
            number_of_nodules=("nodule_id", "count"),
            mean_target_probability=("target_probability", "mean"),
            std_target_probability=("target_probability", "std"),
        )
    )
    curve_points.to_csv(output_directory / "fidelity_curve_points.csv", index=False)

    fidelity_summary = (
        selected_nodule.groupby(
            ["xai_method", "layer", "blur_name", "blur_sigma", "blur_kernel_size"],
            as_index=False,
        )
        .agg(
            number_of_nodules=("nodule_id", "count"),
            number_of_slices=("number_of_slices", "sum"),
            mean_deletion_auc=("mean_slice_deletion_auc", "mean"),
            std_deletion_auc=("mean_slice_deletion_auc", "std"),
            mean_relative_deletion_auc=("mean_slice_relative_deletion_auc", "mean"),
            mean_probability_drop_at_5_percent=("mean_probability_drop_at_5_percent", "mean"),
            mean_probability_drop_at_10_percent=("mean_probability_drop_at_10_percent", "mean"),
            mean_probability_drop_at_20_percent=("mean_probability_drop_at_20_percent", "mean"),
            mean_probability_drop_at_30_percent=("mean_probability_drop_at_30_percent", "mean"),
            mean_probability_drop_at_40_percent=("mean_probability_drop_at_40_percent", "mean"),
            mean_probability_drop_at_50_percent=("mean_probability_drop_at_50_percent", "mean"),
        )
    )
    fidelity_summary.to_csv(output_directory / "fidelity_summary.csv", index=False)
    figure_paths = {
        method: plot_fidelity_curves(
            curve_points,
            method,
            output_directory / f"fidelity_deletion_curve_{method}.png",
        )
        for method in METHODS
    }
    fidelity_results[scope_name] = (fidelity_summary, figure_paths)

print(f"Fidelity outputs: {fidelity_root}")

## 7.3 Fidelity Tables and Figures

In [ ]:
for scope_name, (summary, figure_paths) in fidelity_results.items():
    print(scope_name)
    display(summary.round(4))
    for method in METHODS:
        display(Image(filename=str(figure_paths[method])))

# 8. Output Summary

In [ ]:
configuration = {
    "experiment_id": EXPERIMENT_ID,
    "model": "multistage_direct_guided_cv_resnet50",
    "held_out_inference_runs": 1,
    "source_scopes": list(SOURCE_SCOPES),
    "evaluation_levels": ["slice", "nodule"],
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "nodule_prediction": "majority vote with mean malignant probability tie-breaking",
    "nodule_roc_pr_probability": "mean malignant probability across slices",
    "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "confidence_level": CONFIDENCE_LEVEL,
    "bootstrap_unit": "patient",
    "top_k_percentages": TOP_K_PERCENTAGES,
    "deletion_percentages": (0.0, *DELETION_PERCENTAGES),
    "guidance_probability_maps": str(PROBABILITY_ROOT),
    "fidelity_guidance_protocol": (
        "perturb CT while keeping the original U-Net probability map fixed"
    ),
    "figure_dpi": FIGURE_DPI,
}
with (DRIVE_OUTPUT_DIR / "analysis_configuration.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(configuration, file, indent=2)
    file.write("\n")

output_files = sorted(
    path.relative_to(DRIVE_OUTPUT_DIR)
    for path in DRIVE_OUTPUT_DIR.rglob("*")
    if path.is_file()
)
output_manifest = pd.DataFrame(
    {
        "relative_path": [str(path) for path in output_files],
        "size_bytes": [
            (DRIVE_OUTPUT_DIR / path).stat().st_size for path in output_files
        ],
    }
)
output_manifest.to_csv(DRIVE_OUTPUT_DIR / "output_manifest.csv", index=False)
display(output_manifest)
print(f"Output directory: {DRIVE_OUTPUT_DIR}")